# 07 - Filling Missing Values (Hands-On)

Every line from `07_Filling_Missing_Values.MD`'s Method A (statistic fill), run for real against `loans.csv`, broken into small enough pieces that each step's own output is visible.

Companion notebook `07b_Ordered_And_Wide_Fill_Practice.ipynb` covers Method B (forward/backward fill) and the `axis=1` case, using two different datasets built specifically for those - `loans.csv` can't demonstrate either one correctly.

In [1]:
import pandas as pd

In [2]:
dataset = pd.read_csv("loans.csv")
dataset.shape

(618, 13)

## Recap: which columns need filling

In [3]:
dataset.isnull().sum()

Loan_ID               0
Gender               13
Married               6
Dependents           15
Education             9
Self_Employed        32
ApplicantIncome       2
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     21
Credit_History       50
Property_Area         9
Loan_Status           0
dtype: int64

## The screenshot's code, tested live

Running the exact line from the course slide against `Gender` first, before touching anything else.

In [5]:
dataset["Gender"].fillna(dataset["Gender"].mode()[0], inplace=True)

/var/folders/07/ymtnct793nj_13sf3p9t0lqh0000gn/T/ipykernel_60747/1350428233.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  dataset["Gender"].fillna(dataset["Gender"].mode()[0], inplace=True)


0        Male
1        Male
2      Female
3        Male
4        Male
        ...  
613    Female
614      Male
615      Male
616      Male
617      Male
Name: Gender, Length: 618, dtype: str

In [4]:
dataset["Gender"].isnull().sum()

np.int64(13)

Still **13** - unchanged. The warning above (`ChainedAssignmentError`) is pandas 3.0 telling you exactly this: `inplace=True` edited a temporary copy under Copy-on-Write, not `dataset` itself.

In [6]:
dataset["Gender"] = dataset["Gender"].fillna(dataset["Gender"].mode()[0])

In [6]:
dataset["Gender"].isnull().sum()

np.int64(13)

**0** now - reassigning instead of using `inplace` is the version that actually works.

## The other gotcha - `select_dtypes(include="object")`

In [8]:
dataset.select_dtypes(include="object").columns

/var/folders/07/ymtnct793nj_13sf3p9t0lqh0000gn/T/ipykernel_60359/638993211.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  dataset.select_dtypes(include="object").columns


Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area', 'Loan_Status'],
      dtype='str')

Works, but with a deprecation warning - pandas 3.0 gave text columns their own `str` dtype.

In [9]:
dataset.select_dtypes(include="str").columns

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area', 'Loan_Status'],
      dtype='str')

Same columns, no warning. This is the version to actually use going forward.

## Filling the rest of the categorical columns

In [8]:
categorical_cols = dataset.select_dtypes(include="str").columns
categorical_cols

Index(['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education',
       'Self_Employed', 'Property_Area', 'Loan_Status'],
      dtype='str')

In [9]:
for col in categorical_cols:
    dataset[col] = dataset[col].fillna(dataset[col].mode()[0])

dataset[categorical_cols].isnull().sum()

Loan_ID          0
Gender           0
Married          0
Dependents       0
Education        0
Self_Employed    0
Property_Area    0
Loan_Status      0
dtype: int64

`Gender` was already filled above, so it's a no-op for that one column; every other categorical column drops to 0 here.

## Numerical columns - median, except `Credit_History`

In [12]:
numerical_cols = dataset.select_dtypes(include=["int64", "float64"]).columns
numerical_cols

Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='str')

In [13]:
dataset["Credit_History"].value_counts()

Credit_History
1.0    475
0.0     93
Name: count, dtype: int64

Only two values, `0.0` and `1.0` - a flag, not a real continuous number.

In [14]:
dataset["Credit_History"].mode()[0]

np.float64(1.0)

The more common of the two - this is what fills `Credit_History`'s gaps, not an average of them.

In [15]:
dataset["ApplicantIncome"].mean(), dataset["ApplicantIncome"].median()

(np.float64(4730.256493506494), np.float64(4325.0))

Mean and median aren't the same number - a sign of skew (a few very high earners pulling the mean up). That gap is exactly why median is the safer default for filling this column, not mean.

In [16]:
numerical_cols = dataset.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    if col == "Credit_History":
        dataset[col] = dataset[col].fillna(dataset[col].mode()[0])
    else:
        dataset[col] = dataset[col].fillna(dataset[col].median())

dataset[numerical_cols].isnull().sum()

ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
dtype: int64

Every numerical column at 0 - `Credit_History` filled with its mode, everything else with its median.

In [17]:
dataset.isnull().sum().sum()

np.int64(0)

## Recap

- Reassignment (`dataset[col] = dataset[col].fillna(...)`), not `inplace=True` - verified live, not just claimed.
- `select_dtypes(include="str")`, not `"object"` - same result, no deprecation warning.
- Categorical columns -> mode. Numerical columns -> median, except a number-that's-really-a-flag (`Credit_History`) -> mode too.
- `0` total missing values left in `dataset` - fully filled, 618 rows x 13 columns.

Next: `07b_Ordered_And_Wide_Fill_Practice.ipynb` for `ffill`/`bfill` and `axis=1`, the two techniques `loans.csv` can't correctly demonstrate.